# NCBI Virus Dataset Creation

Author: Alexander Maksiaev

Purpose: Create dataset using NCBI, by de-duplicating from Andersen/GISAID, and relabeling sequences. 

In [1]:
# Housekeeping

import os
import pandas as pd
import dateutil
import re
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [2]:
# Paths

# Dates
start_date = "04-14-2025"
end_date = "06-13-2025"
date_range = start_date + "--" + end_date
update_date = "06-16-2025"
genotypes = ["D1.1", "B3.2", "B3.6", "B3.13", "A3"]

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
downloads = home + "NCBI_Virus/downloads/" + date_range + "/"
temp_files = home + "NCBI_Virus/temp/"
complete_files = home + "NCBI_Virus/complete/" + date_range + "/" # + "_B3_13/"
# gisaid_andersen = home + "Cats/Datasets/GISAID_Andersen/GISAID_Andersen_cats_only_6_5_2025/"
# combined_files = home + "Cats/Datasets/GISAID_Andersen_NCBI_Virus/cat_genotypes_" + update_date + "/" # 04-14-2025--06-05-2025"
gisaid_andersen = home + "Combinations/GISAID_Andersen/" # + date_range + "_" # B3_13/"
combined_files = home + "Combinations/GISAID_Andersen_NCBI_Virus/" # + update_date + "_B3_13/"
references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
# references = "C:/Users/maksi/Documents/Statistics/projects/Avian_Flu/references/"

os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

os.chdir(downloads)

## De-Duplication

In [3]:
metadata = pd.read_csv("sequences.csv")

print(len(metadata))

# Make sure we only have completed sequences -- 8 segments each 

metadata_counts = metadata.groupby(metadata.SRA_Accession, as_index=False).size()

print(metadata_counts)

metadata_counted = metadata.merge(metadata_counts, on="SRA_Accession")

# Only keep those with size=8

metadata_complete_segs = metadata_counted[metadata_counted["size"] >= 8] # May have duplicates

metadata_complete_segs = metadata_complete_segs.drop_duplicates(subset="GenBank_Title", keep="first") # Get rid of duplicate segments

# Now only accept == 8 segments

metadata_segments = metadata_complete_segs[metadata_complete_segs["size"] == 8]

metadata_segments

5751
    SRA_Accession  size
0     SRR30811257     8
1     SRR31254292     8
2     SRR31254312     8
3     SRR31254355     8
4     SRR31254366     8
..            ...   ...
644   SRR33508855     8
645   SRR33508856     8
646   SRR33508857     8
647   SRR33508858     8
648   SRR33508859     8

[649 rows x 2 columns]


,Accession,Organism_Name,GenBank_RefSeq,Assembly,SRA_Accession,Submitters,Organization,Org_location,Release_Date,Isolate,...,Segment,Geo_Location,USA,Host,Tissue_Specimen_Source,Collection_Date,BioSample,BioProject,GenBank_Title,size
0,PV709274.1,Influenza A virus,GenBank,NaN,SRR33369776,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,2025-05-30,25-012798-008-original,...,1,USA: NY,NY,Anatidae,feces,2025-04-16,SAMN48199761,PRJNA980729,Influenza A virus (A/Duck/NY/25-012798-008-ori...,8
1,PV709275.1,Influenza A virus,GenBank,NaN,SRR33369776,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,2025-05-30,25-012798-008-original,...,2,USA: NY,NY,Anatidae,feces,2025-04-16,SAMN48199761,PRJNA980729,Influenza A virus (A/Duck/NY/25-012798-008-ori...,8
2,PV709276.1,Influenza A virus,GenBank,NaN,SRR33369776,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,2025-05-30,25-012798-008-original,...,3,USA: NY,NY,Anatidae,feces,2025-04-16,SAMN48199761,PRJNA980729,Influenza A virus (A/Duck/NY/25-012798-008-ori...,8
3,PV709277.1,Influenza A virus,GenBank,NaN,SRR33369776,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,2025-05-30,25-012798-008-original,...,4,USA: NY,NY,Anatidae,feces,2025-04-16,SAMN48199761,PRJNA980729,Influenza A virus (A/Duck/NY/25-012798-008-ori...,8
4,PV709278.1,Influenza A virus,GenBank,NaN,SRR33369776,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,2025-05-30,25-012798-008-original,...,5,USA: NY,NY,Anatidae,feces,2025-04-16,SAMN48199761,PRJNA980729,Influenza A virus (A/Duck/NY/25-012798-008-ori...,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5187,PV494554.1,Influenza A virus,GenBank,GCA_049648785.1,SRR32415223,"Aufderhar,M., Franzen,K., Killian,M., Lantz,K....",USDA Animal Plant Health Inspection Service-Na...,USA,2025-04-14,25-003975-001-original,...,4,USA: CA,CA,Bos taurus,mammalian milk,2025-01-29,SAMN46921925,PRJNA1102327,Influenza A virus (A/cattle/CA/25-003975-001-o...,8
5188,PV494555.1,Influenza A virus,GenBank,GCA_049648785.1,SRR32415223,"Aufderhar,M., Franzen,K., Killian,M., Lantz,K....",USDA Animal Plant Health Inspection Service-Na...,USA,2025-04-14,25-003975-001-original,...,5,USA: CA,CA,Bos taurus,mammalian milk,2025-01-29,SAMN46921925,PRJNA1102327,Influenza A virus (A/cattle/CA/25-003975-001-o...,8
5189,PV494556.1,Influenza A virus,GenBank,GCA_049648785.1,SRR32415223,"Aufderhar,M., Franzen,K., Killian,M., Lantz,K....",USDA Animal Plant Health Inspection Service-Na...,USA,2025-04-14,25-003975-001-original,...,6,USA: CA,CA,Bos taurus,mammalian milk,2025-01-29,SAMN46921925,PRJNA1102327,Influenza A virus (A/cattle/CA/25-003975-001-o...,8
5190,PV494557.1,Influenza A virus,GenBank,GCA_049648785.1,SRR32415223,"Aufderhar,M., Franzen,K., Killian,M., Lantz,K....",USDA Animal Plant Health Inspection Service-Na...,USA,2025-04-14,25-003975-001-original,...,7,USA: CA,CA,Bos taurus,mammalian milk,2025-01-29,SAMN46921925,PRJNA1102327,Influenza A virus (A/cattle/CA/25-003975-001-o...,8


In [5]:
# De-duplicate from Andersen/GISAID using isolate

# If even one isolate in this list exists in the NCBI Virus dataframe, remove it from NCBI Virus dataframe
gisaid_isolates = []
andersen_isolates = []
# Grab files
for genotype in genotypes:
    # print(gisaid_andersen + genotype.replace(".", "_") + "/")
    for dirpath, dirs, files in os.walk(gisaid_andersen + genotype.replace(".", "_") + "/" + date_range + "_" + genotype.replace(".", "_") + "/"):
        for file in files:
            file_name = os.path.join(dirpath, file)
            # print(file_name)
            if ".fasta" in file_name:
                fasta_file = fasta_df(file_name, states_ref) # Convert fasta file to dataframe
                print(fasta_file)
                # isolates = fasta_file["Isolate_Name"].apply(lambda x: x.split("/")[3])
                isolates = fasta_file["Isolate_Id"]
                for index, value in isolates.items():
                    gisaid_isolates.append(value)

                isolate_ids = fasta_file["Identifier"]
                for index, value in isolates.items():
                    if "SRR" in value:
                        andersen_isolates.append(value)
            
        break 

gisaid_isolates = list(set(gisaid_isolates))

print(len(gisaid_isolates))
print(gisaid_isolates)

['A', 'CATTLE', 'USA', '25-006783-001', '2025']
['A', 'QUAIL', 'USA', '25-006346-009', '2025']
['A', 'DUCK', 'USA', '25-007078-002', '2025']
['A', 'DUCK', 'USA', '25-007078-001', '2025']
['A', 'DUCK', 'USA', '25-006674-002', '2025']
['A', 'CHICKEN', 'USA', '25-007056-001', '2025']
['A', 'CHICKEN', 'USA', '25-006346-008', '2025']
['A', 'CHICKEN', 'USA', '25-006346-007', '2025']
['A', 'CHICKEN', 'USA', '25-006346-004', '2025']
['A', 'DUCK', 'USA', '25-004480-001', '2025']
['A', 'BALD EAGLE', 'USA', '25-005869-001', '2025']
['A', 'CAT', 'USA', '25-006047-001', '2025']
['A', 'CANADA GOOSE', 'USA', '25-004415-006', '2025']
['A', 'CANADA GOOSE', 'USA', '25-004376-001', '2025']
['A', 'CANADA GOOSE', 'USA', '25-002735-001', '2025']
['A', 'MALLARD', 'USA', '25-005927-005', '2025']
['A', 'MALLARD', 'USA', '25-005882-005', '2025']
['A', 'HOODED MERGANSER', 'USA', '25-005880-001', '2025']
['A', 'GLAUCOUS GULL', 'USA', '25-004499-006', '2025']
['A', 'GADWALL', 'USA', '25-005882-002', '2025']
['A', 

In [7]:
# Remove duplicates from GISAID/Andersen

print(len(metadata_segments))

count = 0
isolate_partial = {}
for value in metadata_segments["Isolate"].values:
    # partial = value.split("_")[-1] # If 25_, get the last bit
    # print(partial)
    digits = value.split("-")
    isolate = ""
    # other = ""
    for d in digits:
        # print(d)
        if len(d) == 6 and d.isnumeric(): # If it's just digits and not one of those weird isolates
            isolate = d + "-"
        elif len(d) == 3 and d.isnumeric():
            isolate = isolate + d
        # elif d.isnumeric() == False: # If it's a weird isolate
        #     other = d + "-"
        # else: # If it's a weird isolate
        #     other = other + d
    # Now add to list to check in Andersen files without doing wild for loops
    if len(isolate) == 10: # If this is a correctly formatted isolate
        # isolates.append(isolate)
        # All headers are followed by sequences
        isolate_partial[value] = isolate
    else: # If this is some other isolate
        isolate_partial[value] = value
    # print(isolate)
# for index, value in metadata_segments["Isolate"].items():




to_remove = andersen_isolates
for value in isolate_partial.keys():
    isolate = isolate_partial[value]
    # print(isolate)
    r = re.compile(".*" + isolate + ".*")
    # print(r)
    if len(list(filter(r.search, gisaid_isolates))) > 0:
        for i in list(filter(r.search, gisaid_isolates)):
            to_remove.append(value) 
    # print(to_remove)
    # if isolate_partial[value] in gisaid_andersen_isolates:
    # if value in gisaid_andersen_isolates:
    #     count += 1
    #     to_remove.append(value)
            # metadata_segments = metadata_segments.drop(index)

print(to_remove)
print(len(to_remove))

for value in to_remove:
    if "SRR" in value:
        metadata_segments = metadata_segments[metadata_segments["SRA_Accession"] != value]
    # print(value)
    # print(isolate_partial[value])
    else:
        metadata_segments = metadata_segments[metadata_segments["Isolate"] != value]

print(len(metadata_segments))
print(len(isolate_partial))
metadata_segments
# print(count)


5192
['25-012798-008-original', '25-009630-002-original', '24-036138-001-original-repeat2', '24-036138-002-original-repeat2', '24-036187-001-original-repeat2', '24-036187-002-original-repeat2', '24-036203-001-original-repeat2', '24-036203-002-original-repeat2', '24-036204-001-original-repeat2', '25-006243-001-original', '25-006243-002-original', '25-006243-005-original', '25-006243-006-original', '25-011907-002-original', '25-011907-003-original', '25-011907-006-original', '25-011907-007-original', '25-011907-008-original', '25-011907-009-original', '25-010517-001-original', '25-010517-002-original', '25-010517-003-original', '25-010517-005-original', '25-012114-001-original', '25-012125-001-original', '25-012132-001-original', '25-011508-001-original-repeat2', '25-009621-002-original', '25-010935-009-original', '25-010935-010-original', '25-010935-011-original', '25-010935-003-original', '25-010935-004-original', '25-010817-001-original', '25-009630-001-original', '25-009647-001-origi

,Accession,Organism_Name,GenBank_RefSeq,Assembly,SRA_Accession,Submitters,Organization,Org_location,Release_Date,Isolate,...,Segment,Geo_Location,USA,Host,Tissue_Specimen_Source,Collection_Date,BioSample,BioProject,GenBank_Title,size
16,PV709290.1,Influenza A virus,GenBank,NaN,SRR30811257,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,2025-05-30,24-027096-002-original,...,1,USA: CA,CA,Bos taurus,mammalian milk,2024-09-12,SAMN43929871,PRJNA1102327,Influenza A virus (A/cattle/CA/24-027096-002-o...,8
17,PV709291.1,Influenza A virus,GenBank,NaN,SRR30811257,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,2025-05-30,24-027096-002-original,...,2,USA: CA,CA,Bos taurus,mammalian milk,2024-09-12,SAMN43929871,PRJNA1102327,Influenza A virus (A/cattle/CA/24-027096-002-o...,8
18,PV709292.1,Influenza A virus,GenBank,NaN,SRR30811257,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,2025-05-30,24-027096-002-original,...,3,USA: CA,CA,Bos taurus,mammalian milk,2024-09-12,SAMN43929871,PRJNA1102327,Influenza A virus (A/cattle/CA/24-027096-002-o...,8
19,PV709293.1,Influenza A virus,GenBank,NaN,SRR30811257,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,2025-05-30,24-027096-002-original,...,4,USA: CA,CA,Bos taurus,mammalian milk,2024-09-12,SAMN43929871,PRJNA1102327,Influenza A virus (A/cattle/CA/24-027096-002-o...,8
20,PV709294.1,Influenza A virus,GenBank,NaN,SRR30811257,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,2025-05-30,24-027096-002-original,...,5,USA: CA,CA,Bos taurus,mammalian milk,2024-09-12,SAMN43929871,PRJNA1102327,Influenza A virus (A/cattle/CA/24-027096-002-o...,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5187,PV494554.1,Influenza A virus,GenBank,GCA_049648785.1,SRR32415223,"Aufderhar,M., Franzen,K., Killian,M., Lantz,K....",USDA Animal Plant Health Inspection Service-Na...,USA,2025-04-14,25-003975-001-original,...,4,USA: CA,CA,Bos taurus,mammalian milk,2025-01-29,SAMN46921925,PRJNA1102327,Influenza A virus (A/cattle/CA/25-003975-001-o...,8
5188,PV494555.1,Influenza A virus,GenBank,GCA_049648785.1,SRR32415223,"Aufderhar,M., Franzen,K., Killian,M., Lantz,K....",USDA Animal Plant Health Inspection Service-Na...,USA,2025-04-14,25-003975-001-original,...,5,USA: CA,CA,Bos taurus,mammalian milk,2025-01-29,SAMN46921925,PRJNA1102327,Influenza A virus (A/cattle/CA/25-003975-001-o...,8
5189,PV494556.1,Influenza A virus,GenBank,GCA_049648785.1,SRR32415223,"Aufderhar,M., Franzen,K., Killian,M., Lantz,K....",USDA Animal Plant Health Inspection Service-Na...,USA,2025-04-14,25-003975-001-original,...,6,USA: CA,CA,Bos taurus,mammalian milk,2025-01-29,SAMN46921925,PRJNA1102327,Influenza A virus (A/cattle/CA/25-003975-001-o...,8
5190,PV494557.1,Influenza A virus,GenBank,GCA_049648785.1,SRR32415223,"Aufderhar,M., Franzen,K., Killian,M., Lantz,K....",USDA Animal Plant Health Inspection Service-Na...,USA,2025-04-14,25-003975-001-original,...,7,USA: CA,CA,Bos taurus,mammalian milk,2025-01-29,SAMN46921925,PRJNA1102327,Influenza A virus (A/cattle/CA/25-003975-001-o...,8


In [8]:
# NCBI Virus Naming Convention Example:
# Influenza A virus |USA: IN|25-006338-001-original|H5N1|2025-02-20|Meleagris gallopavo|GenBank|SRR33124721|SAMN47941411|PRJNA980729|Influenza A virus (A/Turkey/IN/25-006338-001-original/2025(H5N1)) segment 1 polymerase PB2 (PB2) gene, complete cds
# Organism_Name | Geo_Location | Isolate | Genotype | Collection_Date | Host | GenBank_RefSeq | SRA_Accession | BioSample | BioProject | GenBank_Title


# print(metadata_segments["GenBank_Title"].iloc[0])

# Get sequences and headers together
headers = []
isolates = []
sras = []
headers_seqs = {}
with open("sequences.fasta") as f:
    lines = f.readlines()
    for num, line in enumerate(lines):
        if line[0] == ">": # If this is a header
            if line.strip() not in headers: # And is not a header we've seen before
                header = line.strip() 
                # print(header)
                split_header = header.split("|")
                # print(split_header)
                headers.append(header) 
                if num < len(lines): # If we're not at the last line
                    # for i, l in enumerate(lines[num + 1:]):
                    i = num
                    sequence = ""
                    # print(lines[i])
                    # print(lines[i + 1])
                    while i < len(lines) - 1 and lines[i + 1][0] != ">": # While the next line is part of a sequence
                        sequence = sequence + lines[i + 1].strip()
                        i += 1
                    headers_seqs[header] = sequence # Add next lines to sequences
                # name = split_header[-1] # Last part gives GenBank name, with segment and isolate
                isolate = split_header[2]
                isolates.append(isolate)
                sra = split_header[7]
                sras.append(sra)
    f.close()



# print(mask_all.all(axis=1))

In [9]:
# Double-check de-duplication
mask = metadata_segments["Isolate"].isin(isolates) # Check if isolate is in fasta
mask_sra = metadata_segments["SRA_Accession"].isin(sras) # Check if sra is in fasta

metadata_masked = metadata_segments[mask] # Some do not have isolates, so check sra too
metadata_masked_sra = metadata_segments[mask_sra]

merged_metadata = metadata_masked.merge(metadata_masked_sra, how="outer")
merged_metadata = merged_metadata.drop_duplicates(keep="first")

print(len(headers_seqs))
print(len(merged_metadata))

5751
3568


In [10]:
# Add sequences to a dataframe 
headers_seqs_df = pd.DataFrame.from_dict(headers_seqs, orient="index")
headers_seqs_df = headers_seqs_df.reset_index()
headers_seqs_df["GenBank_Title"] = headers_seqs_df["index"].apply(lambda x: x[1:].split("|")[-1])
headers_seqs_df = headers_seqs_df.rename(columns={0: "Sequence", "index": "Header"})

# Add sequences to the dataframe
metadata_seqs = merged_metadata.merge(headers_seqs_df, on="GenBank_Title", how="left")

print(metadata_seqs)


       Accession      Organism_Name GenBank_RefSeq         Assembly  \
0     PV493074.1  Influenza A virus        GenBank  GCA_049650035.1   
1     PV493075.1  Influenza A virus        GenBank  GCA_049650035.1   
2     PV493076.1  Influenza A virus        GenBank  GCA_049650035.1   
3     PV493077.1  Influenza A virus        GenBank  GCA_049650035.1   
4     PV493078.1  Influenza A virus        GenBank  GCA_049650035.1   
...          ...                ...            ...              ...   
3563  PV709565.1  Influenza A virus        GenBank              NaN   
3564  PV709566.1  Influenza A virus        GenBank              NaN   
3565  PV709567.1  Influenza A virus        GenBank              NaN   
3566  PV709568.1  Influenza A virus        GenBank              NaN   
3567  PV709569.1  Influenza A virus        GenBank              NaN   

     SRA_Accession                                         Submitters  \
0      SRR32512883  Aufderhar,M., Franzen,K., Killian,M., Lantz,K....   
1

In [11]:
# Create FASTA files per Header

metadata_seqs["Partial_Header"] = metadata_seqs["Header"].apply(lambda x: "|".join(x.split("|")[:-1]))

unique_partial_headers = []

# Each header should repeat 8 times
for partial_header in metadata_seqs["Partial_Header"].values:
    if partial_header not in unique_partial_headers: # Only use the first occurence
        unique_partial_headers.append(partial_header)

print(len(unique_partial_headers)) # Sanity check

446


In [12]:
df_list = []
for partial_header in unique_partial_headers:
    # Get smaller dataframe
    df = metadata_seqs[metadata_seqs["Partial_Header"] == partial_header]
    df = df.sort_values(by="Segment")
    df = df[["Header", "Sequence"]]
    df = df.rename(columns={"Header": "full_header", "Sequence": "sequence"}) # For utils function
    df["full_header"] = df["full_header"].apply(lambda x: ">" + x.split("|")[-1].replace(" ", "_"))
    # Make sure there are 8 segments
    if len(df) == 8:
        df_list.append(df)

In [13]:
print(df_list[0])
print(len(df_list))

                                         full_header  \
0  >Influenza_A_virus_(A/chicken/IN/25-005339-003...   
1  >Influenza_A_virus_(A/chicken/IN/25-005339-003...   
2  >Influenza_A_virus_(A/chicken/IN/25-005339-003...   
3  >Influenza_A_virus_(A/chicken/IN/25-005339-003...   
4  >Influenza_A_virus_(A/chicken/IN/25-005339-003...   
5  >Influenza_A_virus_(A/chicken/IN/25-005339-003...   
6  >Influenza_A_virus_(A/chicken/IN/25-005339-003...   
7  >Influenza_A_virus_(A/chicken/IN/25-005339-003...   

                                            sequence  
0  ATGGAGAGAATAAAAGAACTGAGAGATCTAATGTCACAGTCTCGCA...  
1  ATGGATGTCAATCCGACTTTACTTTTCTTAAAAGTGCCAGCGCAAG...  
2  ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGATCGTCGAGC...  
3  ATGGAAAACATAGTACTTCTTCTTGCAATAATTAGCCTTGTTAAAA...  
4  ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...  
5  ATGAATCCAAATCAAAGGATAATAACCACTGGATCAATCTGTATGG...  
6  ATGAGTCTTCTAACCGAGGTCGAAACGTACGTTCTCTCTATCGTCC...  
7  ATGGATTCCAACACTGTGTCAAGCTTTCAGGTAGACTGCTTTCTTT...  


In [14]:
# Make fasta files. Finally

os.chdir(temp_files)

for df in df_list:
    # print(df)
    file_name = df["full_header"].apply(lambda x: x.split("|")[-1]).values[0] + "_temp.fasta"
    # Forbidden characters
    for c in [">", "/", "|", " ", ":", ",", "(", ")", "-"]:
        file_name = file_name.replace(c, "")
    # print(file_name)
    output_file = open(temp_files + file_name, "w")

    for index, row in df.iterrows():
        name = df.loc[index, "full_header"]
        # print(name)
        sequence = df.loc[index, "sequence"]
        # print(sequence)
        # break 
    # First is header, second is sequence
        output_file.write(name + "\n")
        output_file.write(sequence + "\n")
    output_file.close()

## Re-Labeling Using GenoFlu

In [15]:
# Merging

os.chdir(downloads)

output_genoflu = pd.read_csv("output.tsv", delimiter="\t")

# print(output_genoflu)

# print(metadata_seqs["Header"])

file_name = metadata_seqs["Header"].apply(lambda x: x.split("|")[-1].replace(" ", "_") + "_temp.fasta")
for c in [">", "/", "|", " ", ":", ",", "(", ")", "-"]:
    file_name = file_name.apply(lambda x: x.replace(c, ""))

metadata_seqs["File Name"] = file_name

metadata_genoflu = metadata_seqs.merge(output_genoflu, how="left", on="File Name")

metadata_genoflu = metadata_genoflu.ffill()



print(metadata_genoflu)


       Accession      Organism_Name GenBank_RefSeq         Assembly  \
0     PV493074.1  Influenza A virus        GenBank  GCA_049650035.1   
1     PV493075.1  Influenza A virus        GenBank  GCA_049650035.1   
2     PV493076.1  Influenza A virus        GenBank  GCA_049650035.1   
3     PV493077.1  Influenza A virus        GenBank  GCA_049650035.1   
4     PV493078.1  Influenza A virus        GenBank  GCA_049650035.1   
...          ...                ...            ...              ...   
3563  PV709565.1  Influenza A virus        GenBank  GCA_050773895.1   
3564  PV709566.1  Influenza A virus        GenBank  GCA_050773895.1   
3565  PV709567.1  Influenza A virus        GenBank  GCA_050773895.1   
3566  PV709568.1  Influenza A virus        GenBank  GCA_050773895.1   
3567  PV709569.1  Influenza A virus        GenBank  GCA_050773895.1   

     SRA_Accession                                         Submitters  \
0      SRR32512883  Aufderhar,M., Franzen,K., Killian,M., Lantz,K....   
1

We want:
* Host
* Geo-Location
* Isolate
* Year
* Collection Date
* Host Type
* Genotype

In [16]:
# Re-Labeling

os.chdir(home)
animals_ref = pd.read_csv("animals_ref.csv")

# Make sure NaN doesn't mess up the whole name
metadata_genoflu = metadata_genoflu.fillna("")
metadata_genoflu["Host"] = metadata_genoflu["Host"].apply(str.lower)

fix_animals_andersen(metadata_genoflu, animals_ref)

metadata_genoflu["Years"] = metadata_genoflu["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y"))

# print(metadata_genoflu["Host_Type"])

names = ">" + metadata_genoflu["SRA_Accession"] + "|A/" + metadata_genoflu["Host"] + "/" + metadata_genoflu["Geo_Location"].apply(lambda x: x.split(": ")[-1]) + "/" + metadata_genoflu["Isolate"] + "/" + metadata_genoflu["Years"].apply(lambda x: str(x)) + "|" + metadata_genoflu["Genotype_x"] + "|" + metadata_genoflu["Geo_Location"].apply(lambda x: x.replace(": ", "-")) + "|" + metadata_genoflu["Collection_Date"] + "|" + metadata_genoflu["Host_Type"] + "|" + metadata_genoflu["Genotype_y"]

metadata_genoflu["Name"] = names

print(metadata_genoflu["Name"])


0       >SRR32512883|A/gallus gallus/IN/25-005339-003-...
1       >SRR32512883|A/gallus gallus/IN/25-005339-003-...
2       >SRR32512883|A/gallus gallus/IN/25-005339-003-...
3       >SRR32512883|A/gallus gallus/IN/25-005339-003-...
4       >SRR32512883|A/gallus gallus/IN/25-005339-003-...
                              ...                        
3563    >SRR33029762|A/bos taurus/ID/25-010514-002-ori...
3564    >SRR33029762|A/bos taurus/ID/25-010514-002-ori...
3565    >SRR33029762|A/bos taurus/ID/25-010514-002-ori...
3566    >SRR33029762|A/bos taurus/ID/25-010514-002-ori...
3567    >SRR33029762|A/bos taurus/ID/25-010514-002-ori...
Name: Name, Length: 3568, dtype: object


## Merge all datasets and de-duplicate again

In [17]:
# # Set up segments

# segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"}

# metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].map(segments)

# # print(metadata_genoflu["Segment_Name"])

# # Separate into several dataframes based on genotype + segment
# segment_genotype_dfs = []
# for segment in segments.values():
#     print(segment)
#     m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
#     print(m_g)
#     for genotype in list(set(m_g["Genotype_y"].values)):
#         if "Not assigned:" not in genotype:
#             df = m_g[(m_g["Genotype_y"] == genotype)] # & (metadata_genoflu["Segment"] == segment)]
#             segment_genotype_dfs.append(df)
#             # pair = genotype + "_" + segment
#             # print(pair)

# print(segment_genotype_dfs[3])

# # print(metadata_genoflu["Header"].apply(lambda x: x.split("|")[-1].split("/")[-1].split(")")[2:]))

In [18]:
# Set up segments

segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"}



metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].map(segments)

# print(metadata_genoflu["Segment_Name"])

# Separate into several dataframes based on genotype + segment
# b313_count = 0
segment_genotype_dfs = []
for segment in segments.values():
    # print(segment)
    m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
    # print(m_g)
    for genotype in list(set(m_g["Genotype_y"].values)):
        # if "B3.13" in genotype:
        if genotype in genotypes:
            # b313_count += 1
            df = m_g[(m_g["Genotype_y"] == genotype)] # & (metadata_genoflu["Segment"] == segment)]
            # b313_count += len(df)
            segment_genotype_dfs.append(df)
            pair = genotype + "_" + segment
            print(pair)

# print(b313_count)
print(segment_genotype_dfs[8])

B3.13_PB2
B3.6_PB2
D1.1_PB2
B3.13_PB1
B3.6_PB1
D1.1_PB1
B3.13_PA
B3.6_PA
D1.1_PA
B3.13_HA
B3.6_HA
D1.1_HA
B3.13_NP
B3.6_NP
D1.1_NP
B3.13_NA
B3.6_NA
D1.1_NA
B3.13_MP
B3.6_MP
D1.1_MP
B3.13_NS
B3.6_NS
D1.1_NS
       Accession      Organism_Name GenBank_RefSeq         Assembly  \
10    PV493084.1  Influenza A virus        GenBank  GCA_049649315.1   
74    PV493148.1  Influenza A virus        GenBank  GCA_049649815.1   
82    PV493156.1  Influenza A virus        GenBank  GCA_049649255.1   
90    PV493164.1  Influenza A virus        GenBank  GCA_049649825.1   
98    PV493172.1  Influenza A virus        GenBank  GCA_049649275.1   
...          ...                ...            ...              ...   
3058  PV625424.1  Influenza A virus        GenBank  GCA_050387085.1   
3066  PV625432.1  Influenza A virus        GenBank  GCA_050387155.1   
3074  PV625440.1  Influenza A virus        GenBank  GCA_050387165.1   
3082  PV625448.1  Influenza A virus        GenBank  GCA_050387175.1   
3090  PV62545

In [20]:
# Create FASTA files

os.chdir(complete_files)

names = []

for df in segment_genotype_dfs:
    if len(df["Genotype_y"].values[0]) > 0:
        file_name = df["Genotype_y"].values[0] + "_" + df["Segment_Name"].values[0] + "_" + update_date + ".fasta"
        output_file = open(complete_files + file_name, "w")

        for index, row in df.iterrows():
            name = df.loc[index, "Name"]
            names.append(name)
            # print(name)
            sequence = df.loc[index, "Sequence"]
            # print(sequence)
            # break 
        # First is header, second is sequence
            output_file.write(name + "\n")
            output_file.write(sequence + "\n")
        output_file.close()

print(len(names))

3120


In [21]:
print(segment_genotype_dfs)

[       Accession      Organism_Name GenBank_RefSeq         Assembly  \
288   PV493362.1  Influenza A virus        GenBank  GCA_049648905.1   
296   PV493370.1  Influenza A virus        GenBank  GCA_049649545.1   
304   PV493378.1  Influenza A virus        GenBank  GCA_049647975.1   
312   PV493386.1  Influenza A virus        GenBank  GCA_049648345.1   
320   PV493394.1  Influenza A virus        GenBank  GCA_049647955.1   
...          ...                ...            ...              ...   
3520  PV709474.1  Influenza A virus        GenBank  GCA_050512555.1   
3528  PV709482.1  Influenza A virus        GenBank  GCA_050512555.1   
3536  PV709490.1  Influenza A virus        GenBank  GCA_050512555.1   
3552  PV709554.1  Influenza A virus        GenBank  GCA_050773895.1   
3560  PV709562.1  Influenza A virus        GenBank  GCA_050773895.1   

     SRA_Accession                                         Submitters  \
288    SRR32512688  Aufderhar,M., Franzen,K., Killian,M., Lantz,K....   


In [23]:
# Concatenate

os.chdir(combined_files)

filenames_gisaid_andersen = []
for genotype in genotypes:
    # print(gisaid_andersen + genotype.replace(".", "_") + "/")
    for dirpath, dirs, files in os.walk(gisaid_andersen + genotype.replace(".", "_") + "/" + date_range + "_" + genotype.replace(".", "_") + "/"):
    # for dirpath, dirs, files in os.walk(gisaid_andersen): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            filenames_gisaid_andersen.append(file_name)
        break 

print(filenames_gisaid_andersen)

filenames_ncbi = []
for dirpath, dirs, files in os.walk(complete_files): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        filenames_ncbi.append(file_name)
    break 

print(filenames_ncbi)

do_not_do_these_partials = []
for ga_file in filenames_gisaid_andersen:
    partial_filename_ga = ga_file.split("_")[-4].split("/")[-1] + "_" + ga_file.split("_")[-3]
    for nv_file in filenames_ncbi:
        partial_filename_nv = nv_file.split("_")[-3].split("/")[-1] + "_" + nv_file.split("_")[-2]
        if partial_filename_ga == partial_filename_nv:
            do_not_do_these_partials.append(partial_filename_ga)
            print(partial_filename_nv)
            filenames = [ga_file, nv_file]
            with open(combined_files + partial_filename_ga + "_combined_" + update_date + ".fasta", 'w') as outfile:
                for fname in filenames:
                    with open(fname) as infile:
                        for line in infile:
                            outfile.write(line)

for ga_file in filenames_gisaid_andersen:
    partial_filename_ga = ga_file.split("_")[-4].split("/")[-1] + "_" + ga_file.split("_")[-3]
    if partial_filename_ga not in do_not_do_these_partials:
        with open(combined_files + partial_filename_ga + "_combined_" + update_date + ".fasta", 'w') as outfile2:
            # for fname in filenames_gisaid_andersen:
            with open(ga_file) as infile1:
                for line in infile1:
                    outfile2.write(line)
    

# lines = 0

# fns_seen = set()
# for filename in filenames_ncbi:
#     print("NCBI_Virus", filename)
#     # lines = 0
#     # filename.split("_")[-5] + "." +   
#     partial_filename = filename.split("_")[-3].split("/")[-1] + "_" + filename.split("_")[-2] 
#     print(partial_filename)
    
#     for fn in filenames_gisaid_andersen:
#         # print(fns_seen)
#         if partial_filename in fn: # and partial_filename not in fns_seen: # If partial filenames match
#             fns_seen.add(fn)
#         #     fns_seen.add(partial_filename)
#             # print("hi", partial_filename)
#             filenames = [filename, fn]
#             with open(combined_files + partial_filename + "_combined_" + update_date + ".fasta", 'w') as outfile:
#                 for fname in filenames:
#                     with open(fname) as infile1:
#                         for line in infile1:
#                             outfile.write(line)

# for fn in filenames_gisaid_andersen: # Extra files
#     if fn not in fns_seen:
#         partial_filename = fn.split("_")[-4].split("/")[-1] + "_" + fn.split("_")[-3] 
#         with open(combined_files + partial_filename + "_combined_" + update_date + ".fasta", 'w') as outfile:
#             for fname in filenames:
#                 with open(fname) as infile1:
#                     for line in infile1:
#                         outfile.write(line)
    
#                             # if line[0] == ">":
#                             #     lines += 1
#             #         infile.close()
#             # outfile.close()
#         # elif partial_filename not in fns_seen: # Save the ones that aren't shared -- if partial filename was not seen before
#         #     # print("hello", partial_filename)
#         #     fns_seen.add(partial_filename)
#         #     with open(combined_files + partial_filename + "_combined_" + update_date + ".fasta", 'w') as outfile2:
#         #         # Copy
#         #         with open(fn) as infile2:
#         #             for line in infile2:
#         #                 outfile2.write(line)
#         # else: # If partial filename doesn't match AND if partial filename has been seen before
#         #     continue 
#         #                 # if line[0] == ">":
#         #                 #     lines += 1
#         #             # infile.close()
#         #         with open(fn) as infile3:
#         #             for line in infile3:
#         #                 outfile2.write(line)
#         #                 # if line[0] == ">":
#                         #     lines += 1
#             #     infile.close()
#         outfile.close()


#     # break 

#     # print(lines)

# # lines = 0
# # Counting
#     with open(combined_files + partial_filename + "_combined_" + update_date + ".fasta", 'r') as outfile:
#         for line in outfile.readlines():
#             if line[0] == ">":
#                 lines += 1

# print(lines)

['C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Andersen/D1_1/04-14-2025--06-13-2025_D1_1/D1.1_HA_combined_06-16-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Andersen/D1_1/04-14-2025--06-13-2025_D1_1/D1.1_MP_combined_06-16-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Andersen/D1_1/04-14-2025--06-13-2025_D1_1/D1.1_NA_combined_06-16-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Andersen/D1_1/04-14-2025--06-13-2025_D1_1/D1.1_NP_combined_06-16-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Andersen/D1_1/04-14-2025--06-13-2025_D1_1/D1.1_NS_combined_06-16-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Andersen/D1_1/04-14-2025--06-13-2025_D1_1/D1.1_PA_combined_06-16-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Anders

In [24]:
# # Mega fasta file
# all_files = []
# for dirpath, dirs, files in os.walk(combined_files):
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         all_files.append(file_name)

# with open(combined_files + "all_files_combined_" + update_date + ".fasta", 'w') as outfile3:
#     for fname in all_files:
#         with open(fname) as infile4:
#             for line in infile4:
#                 outfile3.write(line)
# #         infile.close()
# # outfile.close()

In [25]:
# Animals 

os.chdir(home)
animals_ref = pd.read_csv("animals_ref.csv")

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genoflu)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['meleagris gallopavo', 'galliformes', 'felis catus', 'bos taurus', 'gallus gallus', 'numididae sp.', 'anatidae']
[]
                avian               cattle        feline   other_mammal  \
0    great_horned_owl            dairy_cow           cat     deer mouse   
1        common_raven               cattle  domestic_cat    house_mouse   
2       cooper's_hawk  cattle milk product     feral_cat          skunk   
3        coopers_hawk          bovine_milk        feline  striped_skunk   
4             peafowl              bovine   domestic-cat     norway rat   
..                ...                  ...           ...            ...   
415         gyrfalcon                  NaN           NaN            NaN   
416  american_goshawk                  NaN           NaN            NaN   
417      king_vulture                  NaN           NaN            NaN   
418     american coot                  NaN           NaN            NaN   
419        barred owl                  NaN           NaN  